# Dataset Standardisation Pipeline

Load a raw dataset, apply the standardisation pipeline using its YAML config,
and inspect the results.

**Standardisation steps:**
1. Lowercase column names (spaces → underscores)
2. Rename key columns (user_id, item_id, timestamp, rating) via schema mapping
3. Remove rows with null IDs
4. Cast IDs to string
5. Cast all datetime-typed columns to `datetime64[ns]`
6. Cast rating to `float32` (interactions only)
7. Exclude columns declared as `exclude`

In [1]:
import sys
from pathlib import Path
import pandas as pd

# Ensure recdata is importable from the project root
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from recdata.loaders.base_loader import load_config, normalize_file_def
from recdata.loaders.file_reader import read_file
from recdata.processing.standardiser import standardise_df

## Configuration

Set the dataset name and YAML config filename below. The data path and config
path are derived automatically.

In [2]:
# ── Change these two variables for each dataset ──
dataset_name = "xwines"          # folder name under D Datasets
yaml_name    = "xwines.yaml" # YAML config filename

# Paths (auto-derived)
data_path = Path("/Users/gethakloppers/Documents/A PhD/D Datasets") / dataset_name
yaml_path = Path("/Users/gethakloppers/Documents/A PhD/C GitRepos/Data-rec/recdata/configs") / yaml_name

print(f"Data path : {data_path}")
print(f"YAML path : {yaml_path}")
print(f"Data exists: {data_path.is_dir()}")
print(f"YAML exists: {yaml_path.is_file()}")

Data path : /Users/gethakloppers/Documents/A PhD/D Datasets/xwines
YAML path : /Users/gethakloppers/Documents/A PhD/C GitRepos/Data-rec/recdata/configs/xwines.yaml
Data exists: True
YAML exists: True


## 1. Load & validate the YAML config

In [3]:
config = load_config(yaml_path)

print(f"Dataset   : {config['dataset_name']}")
print(f"Domain    : {config.get('domain', 'N/A')}")
print(f"Version   : {config.get('version', 'N/A')}")
print()

# Show declared files
for role in ("interactions", "items", "users"):
    file_def = normalize_file_def(config.get("files", {}).get(role))
    if file_def:
        print(f"  {role:15s} → {file_def['filename']}")
    else:
        print(f"  {role:15s} → (none)")

Dataset   : xwines
Domain    : ecommerce_and_retail
Version   : N/A

  interactions    → XWines_Full_21M_ratings.csv
  items           → XWines_Full_100K_wines.csv
  users           → (none)


## 2. Load raw files

In [4]:
raw_dfs: dict[str, pd.DataFrame] = {}

for role in ("interactions", "items", "users"):
    file_def = normalize_file_def(config.get("files", {}).get(role))
    if file_def is None:
        print(f"[{role}] No file declared — skipping.")
        continue

    filepath = data_path / file_def["filename"]
    if not filepath.exists():
        # Try with archive if specified
        archive = file_def.get("archive")
        if archive:
            filepath = data_path / archive
            print(f"[{role}] Reading '{file_def['filename']}' from archive '{archive}'...")
            df = read_file(filepath, inner_filename=file_def["filename"])
        else:
            print(f"[{role}] ✗ File not found: {filepath}")
            continue
    else:
        print(f"[{role}] Reading '{filepath.name}'...")
        df = read_file(filepath)

    raw_dfs[role] = df
    print(f"         → {len(df):,} rows × {len(df.columns)} cols")
    print(f"         → Columns: {list(df.columns)}")
    print()

[interactions] Reading 'XWines_Full_21M_ratings.csv' from archive 'XWines_Full_21M_ratings.zip'...
         → 21,013,536 rows × 6 cols
         → Columns: ['RatingID', 'UserID', 'WineID', 'Vintage', 'Rating', 'Date']

[items] Reading 'XWines_Full_100K_wines.csv' from archive 'XWines_Full_100K_wines.zip'...
         → 100,646 rows × 17 cols
         → Columns: ['WineID', 'WineName', 'Type', 'Elaborate', 'Grapes', 'Harmonize', 'ABV', 'Body', 'Acidity', 'Code', 'Country', 'RegionID', 'RegionName', 'WineryID', 'WineryName', 'Website', 'Vintages']

[users] No file declared — skipping.


## 3. Apply standardisation

In [5]:
std_dfs: dict[str, pd.DataFrame] = {}
all_warnings: list[str] = []

for role, df_raw in raw_dfs.items():
    print(f"\n{'─' * 50}")
    print(f"Standardising: {role.upper()}")
    print(f"{'─' * 50}")

    df_std, warnings = standardise_df(df_raw, config, role)
    std_dfs[role] = df_std
    all_warnings.extend(warnings)

    print(f"  Before: {len(df_raw):,} rows × {len(df_raw.columns)} cols")
    print(f"  After : {len(df_std):,} rows × {len(df_std.columns)} cols")
    print(f"  Columns: {list(df_std.columns)}")

    if warnings:
        print(f"\n  Warnings:")
        for w in warnings:
            print(f"    ⚠ {w}")


──────────────────────────────────────────────────
Standardising: INTERACTIONS
──────────────────────────────────────────────────


/Users/gethakloppers/Documents/A PhD/C GitRepos/Data-rec/recdata/processing/standardiser.py:381: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df[col] = pd.to_datetime(series, infer_datetime_format=True, errors="coerce")


  Before: 21,013,536 rows × 6 cols
  After : 21,013,536 rows × 5 cols
  Columns: ['user_id', 'item_id', 'vintage', 'rating', 'timestamp']

──────────────────────────────────────────────────
Standardising: ITEMS
──────────────────────────────────────────────────
  Before: 100,646 rows × 17 cols
  After : 100,646 rows × 17 cols
  Columns: ['item_id', 'winename', 'type', 'elaborate', 'grapes', 'harmonize', 'abv', 'body', 'acidity', 'code', 'country', 'regionid', 'regionname', 'wineryid', 'wineryname', 'website', 'vintages']


## 4. Inspect results

In [6]:
for role, df in std_dfs.items():
    print(f"\n{'═' * 50}")
    print(f"  {role.upper()} — {len(df):,} rows × {len(df.columns)} cols")
    print(f"{'═' * 50}")
    print()
    display(df.head())
    print()
    display(df.dtypes.to_frame("dtype"))


══════════════════════════════════════════════════
  INTERACTIONS — 21,013,536 rows × 5 cols
══════════════════════════════════════════════════



,user_id,item_id,vintage,rating,timestamp
0,1604441,136103,1950,4.0,2019-10-14 11:20:52
1,1291483,136103,1950,5.0,2019-11-28 03:36:33
2,1070605,104036,1950,5.0,2017-12-28 10:15:55
3,1080181,144864,1950,5.0,2016-06-23 02:16:22
4,1834379,111430,1950,5.0,2021-05-16 17:58:14


,dtype
user_id,object
item_id,object
vintage,object
rating,float32
timestamp,datetime64[ns]



══════════════════════════════════════════════════
  ITEMS — 100,646 rows × 17 cols
══════════════════════════════════════════════════



,item_id,winename,type,elaborate,grapes,harmonize,abv,body,acidity,code,country,regionid,regionname,wineryid,wineryname,website,vintages
0,100001,Espumante Moscatel,Sparkling,Varietal/100%,['Muscat/Moscato'],"['Pork', 'Rich Fish', 'Shellfish']",7.5,Medium-bodied,High,BR,Brazil,1001,Serra Gaúcha,10001,Casa Perini,http://www.vinicolaperini.com.br,"[2020, 2019, 2018, 2017, 2016, 2015, 2014, 201..."
1,100002,Ancellotta,Red,Varietal/100%,['Ancellotta'],"['Beef', 'Barbecue', 'Codfish', 'Pasta', 'Pizz...",12.0,Medium-bodied,Medium,BR,Brazil,1001,Serra Gaúcha,10001,Casa Perini,http://www.vinicolaperini.com.br,"[2016, 2015, 2014, 2013, 2012, 2011, 2010, 200..."
2,100003,Cabernet Sauvignon,Red,Varietal/100%,['Cabernet Sauvignon'],"['Beef', 'Lamb', 'Poultry']",12.0,Full-bodied,High,BR,Brazil,1001,Serra Gaúcha,10002,Castellamare,https://www.emporiocastellamare.com.br,"[2021, 2020, 2019, 2018, 2017, 2016, 2015, 201..."
3,100004,Virtus Moscato,White,Varietal/100%,['Muscat/Moscato'],['Sweet Dessert'],12.0,Medium-bodied,Medium,BR,Brazil,1001,Serra Gaúcha,10003,Monte Paschoal,http://www.montepaschoal.com.br,"[2021, 2020, 2019, 2018, 2017, 2016, 2015, 201..."
4,100005,Maison de Ville Cabernet-Merlot,Red,Assemblage/Bordeaux Red Blend,"['Cabernet Sauvignon', 'Merlot']","['Beef', 'Lamb', 'Game Meat', 'Poultry']",11.0,Full-bodied,Medium,BR,Brazil,1001,Serra Gaúcha,10000,Aurora,http://www.vinicolaaurora.com.br,"[2021, 2020, 2019, 2018, 2017, 2016, 2015, 201..."


,dtype
item_id,object
winename,object
type,object
elaborate,object
grapes,object
harmonize,object
abv,float64
body,object
acidity,object
code,object


## 5. Summary statistics

In [ ]:
if "interactions" in std_dfs:
    inter = std_dfs["interactions"]
    print(f"Interactions: {len(inter):,} rows")
    if "user_id" in inter.columns:
        print(f"  Unique users: {inter['user_id'].nunique():,}")
    if "item_id" in inter.columns:
        print(f"  Unique items: {inter['item_id'].nunique():,}")
    if "rating" in inter.columns:
        print(f"  Rating range: {inter['rating'].min():.1f} – {inter['rating'].max():.1f}")
        print(f"  Rating mean : {inter['rating'].mean():.2f}")
    if "timestamp" in inter.columns:
        print(f"  Time range  : {inter['timestamp'].min()} — {inter['timestamp'].max()}")

if "items" in std_dfs:
    print(f"\nItems: {len(std_dfs['items']):,} rows")

if "users" in std_dfs:
    print(f"Users: {len(std_dfs['users']):,} rows")

print(f"\nTotal warnings: {len(all_warnings)}")